In [2]:
!pip install -q sentence-transformers pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 21.0 MB/s eta 0:00:00


In [3]:
import urllib.request
import json

# Direct URL to the raw uncompressed JSONL metadata file on Hugging Face
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_All_Beauty.jsonl"

def stream_amazon_data(file_url):
    """Streams and parses a JSONL file directly over HTTP."""
    req = urllib.request.Request(file_url)

    with urllib.request.urlopen(req) as response:
        for line in response:
            yield json.loads(line)

# Initialize the generator
dataset_stream = stream_amazon_data(url)

# Print the first item to verify
first_item = next(dataset_stream)
print("Streaming verified! Here is the first product:")
print(json.dumps(first_item, indent=2))

Streaming verified! Here is the first product:
{
  "main_category": "All Beauty",
  "title": "Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)",
  "average_rating": 4.8,
  "rating_number": 10,
  "features": [],
  "description": [],
  "price": null,
  "images": [
    {
      "thumb": "https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg",
      "large": "https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg",
      "variant": "MAIN",
      "hi_res": null
    },
    {
      "thumb": "https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg",
      "large": "https://m.media-amazon.com/images/I/41w2yznfuZL.jpg",
      "variant": "PT01",
      "hi_res": "https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg"
    }
  ],
  "videos": [],
  "store": "Howard Products",
  "categories": [],
  "details": {
    "Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds",
    "UPC": "617390882781"
  },
  "parent_asin": "B01CUPMQZE",
  "bought_together": null
}


In [4]:
from sentence_transformers import SentenceTransformer

# Initialize the model (using a lightweight, fast, yet highly accurate model)
print("Loading the SentenceTransformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully!")

# Let's take 3 items from our stream to create quick sample texts
temp_stream = stream_amazon_data(url)
sample_texts = []

for i in range(3):
    item = next(temp_stream)
    title = item.get("title", "")
    features = " ".join(item.get("features", [])) if item.get("features") else ""
    text_chunk = f"Product: {title} | Features: {features}"
    sample_texts.append(text_chunk)
    print(f"Sample {i+1}: {text_chunk[:100]}...")

# Generate embeddings
print("\nEmbedding the sample products...")
embeddings = model.encode(sample_texts)

# Print shapes
for i, emb in enumerate(embeddings):
    print(f"Embedding {i+1} shape: {emb.shape} (First 5 values: {emb[:5]}) ")

Loading the SentenceTransformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Sample 1: Product: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack) | Features: ...
Sample 2: Product: Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Frui...
Sample 3: Product: Eye Patch Black Adult with Tie Band (6 Per Pack) | Features: ...

Embedding the sample products...
Embedding 1 shape: (384,) (First 5 values: [-0.07242666  0.11621774 -0.00239776  0.00025719  0.02675459]) 
Embedding 2 shape: (384,) (First 5 values: [-0.06121339  0.01868444 -0.0153756   0.0314416   0.04293544]) 
Embedding 3 shape: (384,) (First 5 values: [-0.02285901  0.01666949 -0.03301628  0.02607836 -0.05138245]) 


In [5]:
from pinecone import Pinecone, ServerlessSpec
import time

# 1. Initialize Pinecone
# Please enter your actual Pinecone API Key below
PINECONE_API_KEY = "pcsk_4xLH3t_2sCAV1VWxL1v6vNLADVsxWUHbxaMhC5bv7KDRx8FD2spNWzN4TCixifMRN3APa1"
pc = Pinecone(api_key=PINECONE_API_KEY)

# 2. Setup Index
index_name = "amazon-recommendations"

if index_name not in pc.list_indexes().names():
    print(f"Creating index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=384,  # Matches the dimension of 'all-MiniLM-L6-v2'
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    # Wait brief moment for index initialization
    time.sleep(5)

index = pc.Index(index_name)
print(f"Pinecone index '{index_name}' is ready!")

# 3. Stream & Ingest Data in Batches
batch_size = 250
batch_ids, batch_texts, batch_metadata = [], [], []
total_upserted = 0

print("Starting batch streaming and ingestion to Pinecone...")

# Fresh stream generator instance
dataset_stream = stream_amazon_data(url)

for item in dataset_stream:
    product_id = item.get("parent_asin")
    if not product_id:
        continue

    title = item.get("title", "")
    features = " ".join(item.get("features", [])) if item.get("features") else ""
    text_chunk = f"Product: {title} | Features: {features}"

    batch_ids.append(product_id)
    batch_texts.append(text_chunk)
    batch_metadata.append({"title": title, "text": text_chunk})

    if len(batch_ids) >= batch_size:
        # Generate embeddings on the fly
        embeddings = model.encode(batch_texts).tolist()

        # Prepare and upsert data structure
        vectors_to_upsert = list(zip(batch_ids, embeddings, batch_metadata))
        index.upsert(vectors=vectors_to_upsert)

        total_upserted += len(batch_ids)
        print(f"Upserted {total_upserted} vectors...")

        # Reset batches
        batch_ids, batch_texts, batch_metadata = [], [], []

        # Limit ingestion for the initial demo run
        if total_upserted >= 2000:
            print("Stopping early for demo limits.")
            break

# Ingest any remaining records
if batch_ids:
    embeddings = model.encode(batch_texts).tolist()
    vectors_to_upsert = list(zip(batch_ids, embeddings, batch_metadata))
    index.upsert(vectors=vectors_to_upsert)
    total_upserted += len(batch_ids)
    print(f"Upserted final batch. Total upserted: {total_upserted}")

print("Ingestion complete!")

Pinecone index 'amazon-recommendations' is ready!
Starting batch streaming and ingestion to Pinecone...
Upserted 250 vectors...
Upserted 500 vectors...
Upserted 750 vectors...
Upserted 1000 vectors...
Upserted 1250 vectors...
Upserted 1500 vectors...
Upserted 1750 vectors...
Upserted 2000 vectors...
Stopping early for demo limits.
Ingestion complete!


In [6]:
def recommend_products(query, top_k=5):
    """Accepts a user query, embeds it, and retrieves the top_k matching products from Pinecone."""
    print(f"Searching recommendations for: '{query}'...\n")

    # 1. Generate embedding for the user query
    query_vector = model.encode(query).tolist()

    # 2. Query Pinecone
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    # 3. Format and display the recommendations
    for idx, match in enumerate(results.get("matches", [])):
        metadata = match.get("metadata", {})
        title = metadata.get("title", "Unknown Product")
        score = match.get("score", 0.0)
        product_id = match.get("id", "N/A")

        print(f"{idx + 1}. {title}")
        print(f"   - Product ID (ASIN): {product_id}")
        print(f"   - Similarity Score: {score:.4f}\n")

# --- Test out the recommendation orchestrator ---
recommend_products("charcoal facial cleanser for glowing skin", top_k=3)

Searching recommendations for: 'charcoal facial cleanser for glowing skin'...

1. Buffing Granules Daily Exfoliating Cleanser - Scrub Away Dead Skin Cells & Stimulates Collagen Production by Newport Cosmeceuticals
   - Product ID (ASIN): B07HPC9TLX
   - Similarity Score: 0.6184

2. KAV PLUS Blackhead Remover Black Mask From Bea Luz, Purifying peel off mask with Activated Charcoal Deep Pore Cleanse for Acne, Oil Control Peel-Off Mask 60g (2.11oz)
   - Product ID (ASIN): B074D39JGY
   - Similarity Score: 0.5885

3. Triple Seven Grooming: Bamboo Charcoal Mask - Organic and Natural, Exfoliating, Pore Unclogging, Acne & Blackhead Reducing, Skin Brightening, Anti-Aging, Deep Facial Cleanser
   - Product ID (ASIN): B098Q4SS34
   - Similarity Score: 0.5798



In [7]:
!pip install -q gradio

In [9]:
import gradio as gr

def recommend_interface(query, num_results):
    """Wrapper function for Gradio to call our recommendation engine."""
    # 1. Generate embedding for the user query
    query_vector = model.encode(query).tolist()

    # 2. Query Pinecone
    results = index.query(
        vector=query_vector,
        top_k=int(num_results),
        include_metadata=True
    )

    # 3. Format matches into HTML cards for a beautiful UI
    html_output = "<div style='display: flex; flex-direction: column; gap: 15px;'>"

    matches = results.get("matches", [])
    if not matches:
        return "<p style='color: #ff4b4b;'>No matching products found. Try another query!</p>"

    for idx, match in enumerate(matches):
        metadata = match.get("metadata", {})
        title = metadata.get("title", "Unknown Product")
        score = match.get("score", 0.0)
        product_id = match.get("id", "N/A")
        percentage_match = f"{score * 100:.1f}%"

        html_output += f"""
        <div style='border: 1px solid #e0e0e0; border-radius: 8px; padding: 15px; background-color: #f9f9f9; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
            <div style='display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;'>
                <span style='background-color: #2196F3; color: white; padding: 3px 8px; border-radius: 12px; font-size: 12px; font-weight: bold;'>#{idx + 1} Recommendation</span>
                <span style='background-color: #4CAF50; color: white; padding: 3px 8px; border-radius: 12px; font-size: 12px; font-weight: bold;'>Match: {percentage_match}</span>
            </div>
            <h3 style='margin: 0 0 8px 0; color: #333; font-family: sans-serif;'>{title}</h3>
            <p style='margin: 0; font-size: 13px; color: #666;'><strong>ASIN:</strong> {product_id}</p>
        </div>
        """
    html_output += "</div>"
    return html_output

# --- Create Gradio Interface ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AI Product-Recommendation Engine")
    gr.Markdown("Enter what you are looking for, and our Sentence Transformer + Pinecone Vector DB system will retrieve the best matching product catalog recommendations instantly!")

    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(
                label="Describe what product you are looking for",
                placeholder="e.g., organic face oil for sensitive skin or activated charcoal mask"
            )
        with gr.Column(scale=1):
            num_results_slider = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Number of Results"
            )

    submit_btn = gr.Button("Find Recommendations", variant="primary")
    output_html = gr.HTML(label="Recommendations")

    # Link button click to the helper function
    submit_btn.click(
        fn=recommend_interface,
        inputs=[query_input, num_results_slider],
        outputs=output_html
    )

# Launch the interface directly in Colab!
demo.launch(share=True)

/tmp/ipykernel_508/4028166735.py:43: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://07003243cb72d4ced6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
